# E03 — Press-Fit & Torque Transmission
*Exam tool — simplified 3-part structure. For full analysis see `04_press_fit.ipynb`.*

---

## Part 1 — Theory Recap

### Diametral Interference
$$\delta = D_{\text{shaft}} - D_{\text{bore}}$$
This is the **diametral** (total) interference. The relative interference is $\delta / D_1$.

### Geometry Factor (Thick-Wall Hub, Lamé)
$$A = \frac{(D_2/D_1)^2 + 1}{(D_2/D_1)^2 - 1}$$
where $D_1$ = bore diameter, $D_2$ = hub outer diameter. Always $A > 1$.

### Time-Dependent Contact Pressure
$$p(t) = \frac{\delta}{D_1} \cdot \frac{E_r(t)}{A + \nu}$$
where $E_r(t)$ is the **relaxation modulus** interpolated from the material table at time $t$.

### Torque Capacity
$$M_t(t) = \frac{\pi}{2} \cdot D_1^2 \cdot L \cdot p(t) \cdot \mu$$

### Log-Log Interpolation for $E_r(t)$
The relaxation modulus table provides 6 points at 0.01, 1, 10, 100, 1000, 10 000 h.
Intermediate values are interpolated on a **log-log** scale:
$$\log E_r(t) = \text{interp}(\log t, \log t_i, \log E_{r,i})$$

### Design Limits for POM
| Check | Limit |
|-------|-------|
| Relative interference $\delta/D_1$ (unfilled POM) | $\leq 3\%$ |
| Relative interference $\delta/D_1$ (glass-filled POM) | $\leq 1\%$ |
| Initial contact pressure (POM yield) | $\leq 35\,\text{MPa}$ |

### Parameter Table
| Symbol | Description | Unit |
|--------|-------------|------|
| $\delta$ | Diametral interference | m |
| $D_1$ | Bore (shaft) diameter | m |
| $D_2$ | Hub outer diameter | m |
| $L$ | Engagement length | m |
| $\nu$ | Poisson's ratio | — |
| $\mu$ | Friction coefficient | — |
| $E_r(t)$ | Relaxation modulus at time $t$ | Pa |
| $M_{t,\text{req}}$ | Minimum required torque | N·m |

---

## ⚠ Common Exam Pitfalls
1. **Diametral vs radial** — $\delta$ is the total (diametral) interference; the formula already divides by $D_1$.
2. **Use end-of-life $E_r$** — the exam asks for torque at the end of service life, NOT at assembly time.
3. **mm to m conversion** — $\delta$, $D_1$, $D_2$, $L$ must all be in metres before calculation.
4. **Geometry factor direction** — $A$ uses $D_2/D_1$ (outer/bore); check $A > 1$ to verify correct order.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Problem Inputs  (edit values here)
# ═══════════════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '..')
import numpy as np

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import RELAXATION_MODULUS, FRICTION_COEFFICIENTS

# ── Material ─────────────────────────────────────────────────────────────────
MAT_KEY = 'POM_unfilled'   # keys: 'POM_unfilled', 'POM_GF30', 'PP', 'PC'
mat     = RELAXATION_MODULUS[MAT_KEY]
nu      = mat['poisson']
mu      = FRICTION_COEFFICIENTS['POM_steel']   # 0.20

# ── Geometry ─────────────────────────────────────────────────────────────────
D_shaft = Q_(20.0, 'mm')        # shaft outer diameter (= bore diameter D1)
D_bore  = Q_(19.8, 'mm')        # hub bore diameter before assembly
D2      = Q_(32.0, 'mm')        # hub outer diameter
L_eng   = Q_(25.0, 'mm')        # engagement length

# ── Service life ─────────────────────────────────────────────────────────────
t_service  = Q_(1000.0, 'hour')
M_required = Q_(3.0, 'N*m')     # minimum torque the joint must transmit

# ── Yield limit for initial pressure check ───────────────────────────────────
POM_YIELD_STRESS_PA = 35e6   # POM (Delrin 100) yield stress [Pa], DuPont datasheet

print(f'Material     : {mat["description"]}')
print(f'Poisson ν    : {nu}')
print(f'Friction μ   : {mu}')


Material     : POM unfilled
Poisson ν    : 0.38
Friction μ   : 0.2


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Functions
# ═══════════════════════════════════════════════════════════════════════════════

def geometry_factor(D1_m, D2_m):
    """Lamé geometry factor A = [(D2/D1)^2 + 1] / [(D2/D1)^2 - 1]. Always > 1."""
    ratio2 = (D2_m / D1_m) ** 2
    A = (ratio2 + 1.0) / (ratio2 - 1.0)
    assert A > 1.0, f'A={A:.3f} <= 1 — check D1/D2 argument order (D1=bore, D2=outer)'
    return A


def relaxation_modulus_interp(t_h, times_h, Er_Pa):
    """Log-log interpolation of E_r(t). Clamps t_h to [min, max] of table."""
    t_min, t_max = min(times_h), max(times_h)
    if t_h < t_min:
        print(f'  NOTE: t={t_h}h < table min ({t_min}h) — using endpoint value')
        t_h = t_min
    elif t_h > t_max:
        print(f'  NOTE: t={t_h}h > table max ({t_max}h) — using endpoint value')
        t_h = t_max
    log_Er = np.interp(np.log10(t_h),
                       np.log10(times_h),
                       np.log10(Er_Pa))
    return 10.0 ** log_Er


def contact_pressure(delta_m, D1_m, Er_Pa_t, A_geo, nu):
    """Contact pressure p(t) = (delta/D1) * Er(t) / (A + nu). Returns Pa."""
    return (delta_m / D1_m) * Er_Pa_t / (A_geo + nu)


def torque_capacity(D1_m, L_m, p_Pa, mu):
    """Torque Mt = (pi/2) * D1^2 * L * p * mu. Returns N·m."""
    return (np.pi / 2.0) * D1_m**2 * L_m * p_Pa * mu


print('Functions defined.')


Functions defined.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Execution
# ═══════════════════════════════════════════════════════════════════════════════

D1_m   = strip_units(D_shaft.to('m'))
D2_m   = strip_units(D2.to('m'))
Db_m   = strip_units(D_bore.to('m'))
L_m    = strip_units(L_eng.to('m'))
t_h    = strip_units(t_service.to('second')) / 3600.0   # hours (not SI seconds)
M_req  = strip_units(M_required.to('N*m'))

delta_m = D1_m - Db_m   # diametral interference

# --- Step 1: Geometry --------------------------------------------------------
A_geo = geometry_factor(D1_m, D2_m)
rel_int_pct = (delta_m / D1_m) * 100.0
print('--- Step 1: Geometry ---')
print(f"  {'D1 (shaft/bore) [mm]':<30}: {D1_m*1e3:.2f}")
print(f"  {'D2 (hub outer) [mm]':<30}: {D2_m*1e3:.2f}")
print(f"  {'D_bore (before assembly) [mm]':<30}: {Db_m*1e3:.3f}")
print(f"  {'δ (diametral interference) [mm]':<30}: {delta_m*1e3:.3f}")
print(f"  {'δ/D1 (relative interference) [%]':<30}: {rel_int_pct:.3f} %")
print(f"  {'A (geometry factor)':<30}: {A_geo:.4f}")
print(f"  {'L (engagement length) [mm]':<30}: {L_m*1e3:.1f}")

# --- Step 2: Relaxation modulus at t=0 (approx) and t_service ----------------
times_h = mat['times_h']
Er_Pa   = mat['Er_Pa']
Er_init = relaxation_modulus_interp(times_h[0], times_h, Er_Pa)   # t≈0 (table min)
Er_svc  = relaxation_modulus_interp(t_h, times_h, Er_Pa)
print()
print('--- Step 2: Relaxation Modulus ---')
print(f"  {'Er at t=' + f'{times_h[0]:.2f}h [GPa]':<30}: {Er_init/1e9:.3f}")
print(f"  {'Er at t=' + f'{t_h:.0f}h [GPa]':<30}: {Er_svc/1e9:.3f}")
print(f"  {'Modulus retention [%]':<30}: {Er_svc/Er_init*100:.1f} %")

# --- Step 3: Contact pressure ------------------------------------------------
p_init = contact_pressure(delta_m, D1_m, Er_init, A_geo, nu)
p_svc  = contact_pressure(delta_m, D1_m, Er_svc,  A_geo, nu)
print()
print('--- Step 3: Contact Pressure ---')
print(f"  {'p at assembly (t≈0) [MPa]':<30}: {p_init/1e6:.3f}")
print(f"  {'p at service life [MPa]':<30}: {p_svc/1e6:.3f}")
print(f"  {'Pressure decay [%]':<30}: {(1 - p_svc/p_init)*100:.1f} %")

# --- Step 4: Torque capacity -------------------------------------------------
Mt_init = torque_capacity(D1_m, L_m, p_init, mu)
Mt_svc  = torque_capacity(D1_m, L_m, p_svc,  mu)
margin  = (Mt_svc / M_req - 1.0) * 100.0
print()
print('--- Step 4: Torque Capacity ---')
print(f"  {'Mt at assembly [N·m]':<30}: {Mt_init:.3f}")
print(f"  {'Mt at service life [N·m]':<30}: {Mt_svc:.3f}")
print(f"  {'M_required [N·m]':<30}: {M_req:.3f}")
print(f"  {'Torque margin at service [%]':<30}: {margin:+.1f} %")


--- Step 1: Geometry ---
  D1 (shaft/bore) [mm]          : 20.00
  D2 (hub outer) [mm]           : 32.00
  D_bore (before assembly) [mm] : 19.800
  δ (diametral interference) [mm]: 0.200
  δ/D1 (relative interference) [%]: 1.000 %
  A (geometry factor)           : 2.2821
  L (engagement length) [mm]    : 25.0

--- Step 2: Relaxation Modulus ---
  Er at t=0.01h [GPa]           : 3.100
  Er at t=1000h [GPa]           : 1.200
  Modulus retention [%]         : 38.7 %

--- Step 3: Contact Pressure ---
  p at assembly (t≈0) [MPa]     : 11.645
  p at service life [MPa]       : 4.508
  Pressure decay [%]            : 61.3 %

--- Step 4: Torque Capacity ---
  Mt at assembly [N·m]          : 36.584
  Mt at service life [N·m]      : 14.162
  M_required [N·m]              : 3.000
  Torque margin at service [%]  : +372.1 %


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Validation
# ═══════════════════════════════════════════════════════════════════════════════

pass_interf = rel_int_pct <= 3.0        # 3% limit for unfilled POM
pass_yield  = p_init <= POM_YIELD_STRESS_PA
pass_torque = Mt_svc >= M_req
overall     = pass_interf and pass_yield and pass_torque

print('--- VALIDATION ---')
print(f"  Relative interference : {rel_int_pct:.3f}%  <=  3.0%"
      f"  |  {'PASS' if pass_interf else 'FAIL'}")
print(f"  Initial pressure      : {p_init/1e6:.2f} MPa  <=  {POM_YIELD_STRESS_PA/1e6:.0f} MPa"
      f"  |  {'PASS' if pass_yield else 'FAIL'}")
print(f"  Torque at service     : {Mt_svc:.3f} N·m  >=  {M_req:.3f} N·m"
      f"  |  {'PASS' if pass_torque else 'FAIL'}")
print(f"  OVERALL               : {'PASS' if overall else 'FAIL'}")
if not pass_torque:
    print()
    print('  Remedies to increase torque at service life:')
    print('    1. Increase interference δ (but check δ/D1 limit)')
    print('    2. Switch to POM_GF30 (higher Er retention)')
    print('    3. Increase engagement length L')
    print('    4. Add surface knurling to increase effective μ')


--- VALIDATION ---
  Relative interference : 1.000%  <=  3.0%  |  PASS
  Initial pressure      : 11.65 MPa  <=  35 MPa  |  PASS
  Torque at service     : 14.162 N·m  >=  3.000 N·m  |  PASS
  OVERALL               : PASS
